# Lecture: Latent Diffusion — the Core Idea behind Stable Diffusion

Every diffusion model so far (C4-2 to C4-4) ran in **pixel space**: the U-Net
denoised full 28×28 images. That is fine for tiny images, but it does **not
scale** — diffusing a 512×512 image directly means every one of the ~1000
denoising steps operates on 262 144 pixels. This is why early diffusion models
were slow and limited in resolution.

**Latent Diffusion Models** (Rombach et al., 2022) — the model family that
*Stable Diffusion* belongs to — solve this with one decisive idea:

> Don't diffuse pixels. First **compress** the image with an autoencoder into a
> small **latent**, run the *entire* diffusion process in that latent space, then
> **decode** the result back to an image.

This is exactly where **Chapter C2 meets Chapter C4**:

$$\underbrace{\text{image} \xrightarrow{\text{encode}} \text{latent}}_{\text{autoencoder (C2)}} \;\xrightarrow{\text{diffusion (C4)}}\; \underbrace{\text{latent} \xrightarrow{\text{decode}} \text{image}}_{\text{autoencoder (C2)}}$$

The diffusion mathematics is **completely unchanged** — same forward process,
same noise-prediction loss, same DDIM sampler, same classifier-free guidance from
C4-4. Only the *thing being diffused* changes: a `4×7×7` latent instead of a
`1×28×28` image (a 16× spatial reduction).

In this notebook we build a **full mini Stable Diffusion** on Fashion-MNIST:

1. Train a compact **autoencoder** to compress images to latents.
2. Train a **class-conditional latent diffusion model** on those latents.
3. Generate specific classes with **classifier-free guidance**, in latent space,
   and decode the results.

Run the following cell only if you are working with Google Colab to copy the required .py files into the root directory. If you are working locally, ignore this cell.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/C4-Diffusion_Models/Diffusion.py ./
!cp AIBIP/C4-Diffusion_Models/AutoEncoder.py ./

### Data Preparation

The same labelled Fashion-MNIST as C4-4 — we keep the labels for conditioning.
Images are normalised to $[-1, 1]$, matching the `Tanh` output of both the
autoencoder and the diffusion model.

In [ ]:
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from torchvision.datasets import FashionMNIST
from AutoEncoder import AutoEncoder
from Diffusion import LatentDDPM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

LATENT_CHANNELS = 4
NUM_CLASSES     = 10
TIMESTEPS       = 1000
BATCH_SIZE      = 128
AE_EPOCHS       = 15
LD_EPOCHS       = 40
LR              = 2e-4

FASHION_CLASSES = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal",  "Shirt",   "Sneaker",  "Bag",   "Ankle boot"
]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_dataset = FashionMNIST(root="./data", train=True, download=True, transform=transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=0, pin_memory=True)

print(f"Training samples: {len(train_dataset)}")

## Stage 1 — The Autoencoder (perceptual compression)

First we train the autoencoder that defines the latent space. The `AutoEncoder`
in `AutoEncoder.py` compresses a `1×28×28` image to a `4×7×7` latent (16× smaller
in spatial area) and reconstructs it. It is trained with a plain reconstruction
(MSE) loss — its only job is to provide a compact, decodable latent.

This is the stage that corresponds to the VAE/autoencoder work in **Chapter C2**.
In real Stable Diffusion this is a much larger VAE, but the principle is identical.

In [ ]:
ae = AutoEncoder(latent_channels=LATENT_CHANNELS, channels=32).to(device)

# Verify the compression shapes
_x = torch.randn(4, 1, 28, 28, device=device)
_z = ae.encode(_x)
print("Image shape: ", tuple(_x.shape))
print("Latent shape:", tuple(_z.shape), "  <- diffusion runs here")
print(f"Spatial compression: {28*28} -> {_z.shape[-1]*_z.shape[-2]} pixels per channel")

In [ ]:
ae_optimizer = optim.Adam(ae.parameters(), lr=LR)
mse = torch.nn.MSELoss()

for epoch in range(AE_EPOCHS):
    ae.train()
    total = 0.0
    for x, _ in train_loader:
        x = x.to(device, non_blocking=True)
        x_rec = ae(x)
        loss = mse(x_rec, x)

        ae_optimizer.zero_grad()
        loss.backward()
        ae_optimizer.step()
        total += loss.item()
    print(f"[AE] Epoch {epoch+1:3d}  recon_loss={total/len(train_loader):.4f}")

ae.save_model(path="models/autoencoder_fashion_mnist.pth")

If you do not want to train the autoencoder, load the pre-trained one.

In [ ]:
ae = AutoEncoder(latent_channels=LATENT_CHANNELS, channels=32).to(device)
#ae.load_model(path="models/autoencoder_fashion_mnist.pth", device=device)            # for running locally
ae.load_model(path="AIBIP/C4-Diffusion_Models/models/autoencoder_fashion_mnist.pth", device=device)  # for running in colab
ae.eval()

### Check the reconstruction quality

Before diffusing in latent space, we must confirm the autoencoder can faithfully
reconstruct images — the diffusion model can never produce better images than the
decoder allows. The top row shows originals, the bottom row reconstructions.

In [ ]:
ae.eval()
x, _ = next(iter(train_loader))
x = x[:8].to(device)
with torch.no_grad():
    x_rec = ae(x)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    axes[0, i].imshow(((x[i].squeeze().cpu() + 1) / 2).clamp(0, 1), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(((x_rec[i].squeeze().cpu() + 1) / 2).clamp(0, 1), cmap="gray")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("original", fontsize=11)
axes[1, 0].set_ylabel("reconstr.", fontsize=11)
plt.suptitle("Autoencoder reconstruction quality", y=1.02)
plt.tight_layout()
plt.show()

## Stage 2 — Pre-compute the latents

With the autoencoder **frozen**, we encode the whole training set **once** into
latents. The diffusion model then trains on these fixed latents — there is no need
to run the encoder during diffusion training, which makes it fast. The latents
carry over the class labels for conditioning.

In [ ]:
ae.eval()
all_latents, all_labels = [], []
with torch.no_grad():
    for x, y in train_loader:
        z = ae.encode(x.to(device)).cpu()
        all_latents.append(z)
        all_labels.append(y)

latents = torch.cat(all_latents)
labels  = torch.cat(all_labels)
print("Pre-computed latents:", tuple(latents.shape))

latent_dataset = TensorDataset(latents, labels)
latent_loader  = DataLoader(latent_dataset, batch_size=BATCH_SIZE, shuffle=True)

## Stage 3 — Train the conditional latent diffusion model

Now the heart of the notebook: a `LatentDDPM` that diffuses the `4×7×7` latents.
It is **the same DDPM** as before — same schedule, same noise-prediction loss,
same class conditioning and label dropout for classifier-free guidance (C4-4) —
only the input is a latent instead of an image. The U-Net (`LatentUNet`) keeps the
spatial size fixed at 7×7 and uses residual blocks instead of pooling, since the
latent is already small.

Because the latents are tiny, this trains **faster** than the pixel-space DDPM in
C4-2 despite producing comparable images — the whole point of latent diffusion.

In [ ]:
ldm = LatentDDPM(latent_channels=LATENT_CHANNELS, timesteps=TIMESTEPS,
                 channels=128, num_classes=NUM_CLASSES).to(device)

n_params = sum(p.numel() for p in ldm.parameters())
print(f"Latent diffusion parameters: {n_params:,}")

In [ ]:
ldm_optimizer = optim.Adam(ldm.parameters(), lr=LR)

history = []
for epoch in range(LD_EPOCHS):
    ldm.train()
    total = 0.0
    for z0, y in latent_loader:
        z0 = z0.to(device, non_blocking=True)
        y  = y.to(device, non_blocking=True)

        loss = ldm.loss(z0, y, p_uncond=0.1)

        ldm_optimizer.zero_grad()
        loss.backward()
        ldm_optimizer.step()
        total += loss.item()

    history.append(total / len(latent_loader))
    print(f"[LDM] Epoch {epoch+1:3d}  loss={history[-1]:.4f}")

ldm.save_model(path="models/latent_ddpm_fashion_mnist.pth")

If you do not want to train, load the pre-trained latent diffusion model.

In [ ]:
ldm = LatentDDPM(latent_channels=LATENT_CHANNELS, timesteps=TIMESTEPS,
                 channels=128, num_classes=NUM_CLASSES).to(device)
#ldm.load_model(path="models/latent_ddpm_fashion_mnist.pth", device=device)            # for running locally
ldm.load_model(path="AIBIP/C4-Diffusion_Models/models/latent_ddpm_fashion_mnist.pth", device=device)  # for running in colab
ldm.eval()

## Generating images: latent diffusion + guidance + decode

The full generation pipeline now mirrors Stable Diffusion exactly:

1. **Sample a latent** with classifier-free guidance for the requested class
   (`ldm.sample_cfg`) — this runs the DDIM reverse process in the `4×7×7` latent
   space.
2. **Decode** the generated latent with the autoencoder to obtain a `28×28` image.

We generate one example of every class. Note that the diffusion model never sees a
pixel — it only ever produces latents, which the decoder turns into images.

In [ ]:
ldm.eval()
ae.eval()

labels = torch.arange(NUM_CLASSES, device=device)
with torch.no_grad():
    z_gen = ldm.sample_cfg(labels, steps=50, guidance_scale=3.0, device=device)
    imgs  = ae.decode(z_gen).cpu()
imgs = (imgs + 1) / 2

fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(imgs[i].squeeze().clamp(0, 1), cmap="gray")
    ax.set_title(FASHION_CLASSES[i], fontsize=8)
    ax.axis("off")
plt.suptitle("Latent diffusion samples, one per class (guidance = 3.0)", y=1.15)
plt.tight_layout()
plt.show()

### Guidance scale in latent space

Exactly as in C4-4, the `guidance_scale` trades class fidelity against diversity —
but now the guidance acts on the **latent** noise prediction, and we decode
afterwards. Sweeping $w$ for a single class shows the same effect, confirming that
CFG is independent of whether we diffuse pixels or latents.

In [ ]:
target_class = 1   # Trouser
n = 8
scales = [0.0, 1.0, 3.0, 6.0]

fig, axes = plt.subplots(len(scales), n, figsize=(14, 7.5))
for row, w in enumerate(scales):
    labels = torch.full((n,), target_class, device=device)
    torch.manual_seed(0)
    with torch.no_grad():
        z_gen = ldm.sample_cfg(labels, steps=50, guidance_scale=w, device=device)
        imgs  = ae.decode(z_gen).cpu()
    imgs = (imgs + 1) / 2
    for col in range(n):
        axes[row, col].imshow(imgs[col].squeeze().clamp(0, 1), cmap="gray")
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(f"w = {w}", rotation=0, labelpad=30, fontsize=11, va="center")
plt.suptitle(f"Guidance scale in latent space — class '{FASHION_CLASSES[target_class]}'", y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

### What we have built — and how it maps to Stable Diffusion

We have assembled a complete, if miniature, **Stable Diffusion**:

| Component here | Stable Diffusion |
|---|---|
| `AutoEncoder` (28×28 ↔ 4×7×7) | VAE (512×512 ↔ 64×64×4 latent) |
| `LatentDDPM` / `LatentUNet` | The latent U-Net |
| Class embedding + null class | CLIP **text** encoder + empty prompt |
| `sample_cfg` (guidance) | Identical CFG sampler |
| DDIM 50-step sampling (C4-3) | DDIM / DPM-Solver sampling |

Every ingredient of Stable Diffusion is present:

- **Latent compression** (this notebook) — diffuse in a small space, not pixels.
- **Conditioning + classifier-free guidance** (C4-4) — steer generation.
- **Fast sampling via DDIM** (C4-3) — generate in tens of steps.
- **The DDPM training objective** (C4-2) — the noise-prediction loss.

The *only* conceptual gap to a real text-to-image model is replacing the class
embedding with a **text encoder** fed via **cross-attention** — which is what the
final notebook (C4-6) demonstrates using a pre-trained Stable Diffusion pipeline.

---
## Try It Yourself — Experiment with Latent Diffusion

Work in pairs. **Predict first, then run, then explain in one sentence.**

**A. The decoder is the ceiling.** Look at the autoencoder reconstruction cell.
The generated images can never be sharper than these reconstructions — why? What
would you change to raise this quality ceiling, and what would it cost?

**B. Latent vs. pixel diffusion.** Compare the `LatentDDPM` training loss/runtime
to the pixel-space `DDPM` from C4-2. Which trains faster, and why? Relate the
answer to the size of the data being diffused ($4{\times}7{\times}7$ vs.
$1{\times}28{\times}28$).

**C. Guidance, decoupled from space.** The $w$-sweep here looks like the one in
C4-4, but guidance now acts on latents. In one sentence, explain why CFG works
identically regardless of whether we diffuse pixels or latents.

**D. Inspect a latent.** Pick a generated latent (`z_gen[i]`) and visualise its 4
channels as small 7×7 heatmaps. Can you see any structure? Why is a latent *not*
expected to look like a recognisable image?